In [56]:
# face crops for gallery using RetinaFace with fallback upscaling

import os
import cv2
import pandas as pd

from tqdm import tqdm
from retinaface import RetinaFace

INPUT_ROOT = "natural_test/gallery"
OUTPUT_ROOT = "natural_test/gallery_cropped"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

failed_images = []
metadata_rows = []

for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)

    if not os.path.isdir(input_dir):
        continue

    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        try:

            img = cv2.imread(image_path)

            if img is None:
                failed_images.append(image_path)
                continue

            h, w = img.shape[:2]

            # -------------------------
            # Detection Attempts
            # -------------------------

            faces = RetinaFace.detect_faces(img)
            scale_used = 1

            if not isinstance(faces, dict) or len(faces) == 0:

                img_2x = cv2.resize(
                    img,
                    None,
                    fx=2,
                    fy=2,
                    interpolation=cv2.INTER_CUBIC,
                )

                faces = RetinaFace.detect_faces(img_2x)

                scale_used = 2

            if not isinstance(faces, dict) or len(faces) == 0:

                img_3x = cv2.resize(
                    img,
                    None,
                    fx=3,
                    fy=3,
                    interpolation=cv2.INTER_CUBIC,
                )

                faces = RetinaFace.detect_faces(img_3x)

                scale_used = 3

            # -------------------------
            # No face detected
            # -------------------------

            if not isinstance(faces, dict) or len(faces) == 0:

                crop = cv2.resize(img, (112, 112))

                save_path = os.path.join(output_dir, image_file)

                cv2.imwrite(save_path, crop)

                metadata_rows.append(
                    {
                        "identity": identity,
                        "image_name": image_file,
                        "det_score": 0.0,
                        "scale_used": 0,
                    }
                )

                failed_images.append(image_path)

                print(f"FULL IMAGE USED | {identity} | {image_file}")

                continue

            # -------------------------
            # Largest face
            # -------------------------

            largest_area = -1
            best_box = None
            best_score = None

            for face_id in faces:

                x1, y1, x2, y2 = faces[face_id]["facial_area"]

                area = (x2 - x1) * (y2 - y1)

                if area > largest_area:

                    largest_area = area

                    best_box = (x1, y1, x2, y2)

                    best_score = float(faces[face_id]["score"])

            if best_box is None:

                failed_images.append(image_path)

                continue

            x1, y1, x2, y2 = best_box

            # -------------------------
            # Scale back coordinates
            # -------------------------

            x1 = int(x1 / scale_used)
            y1 = int(y1 / scale_used)

            x2 = int(x2 / scale_used)
            y2 = int(y2 / scale_used)

            # -------------------------
            # Padding
            # -------------------------

            pad = 20

            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)

            x2 = min(w, x2 + pad)
            y2 = min(h, y2 + pad)

            crop = img[y1:y2, x1:x2]

            if crop.size == 0:

                crop = cv2.resize(img, (112, 112))

            else:

                crop = cv2.resize(crop, (112, 112))

            save_path = os.path.join(output_dir, image_file)

            cv2.imwrite(save_path, crop)

            metadata_rows.append(
                {
                    "identity": identity,
                    "image_name": image_file,
                    "det_score": best_score,
                    "scale_used": scale_used,
                }
            )

            print(
                f"{identity} | "
                f"{image_file} | "
                f"Detection Score={best_score:.6f} | "
                f"Scale={scale_used}x"
            )

        except Exception as e:

            print(f"\nERROR: {image_path}")
            print(e)

            failed_images.append(image_path)

# -------------------------
# Save Logs
# -------------------------

pd.DataFrame({"failed_image": failed_images}).to_csv(
    "gallery_failed_detections.csv",
    index=False,
)

pd.DataFrame(metadata_rows).to_csv(
    "gallery_detection_scores.csv",
    index=False,
)

print("\nGallery Cropping Done")

print("Successful Samples:", len(metadata_rows))
print("Failed Samples:", len(failed_images))

  0%|          | 0/99 [00:00<?, ?it/s]

Zendaya | Zendaya_001.jpg | Detection Score=0.988055 | Scale=1x
Zendaya | Zendaya_002.jpg | Detection Score=0.998676 | Scale=1x
Zendaya | Zendaya_003.jpg | Detection Score=0.999260 | Scale=1x
FULL IMAGE USED | Zendaya | Zendaya_004.jpg


  1%|          | 1/99 [00:19<31:12, 19.10s/it]

FULL IMAGE USED | Zendaya | Zendaya_005.jpg
FULL IMAGE USED | Elon_Musk | Elon_Musk_001.jpg
FULL IMAGE USED | Elon_Musk | Elon_Musk_002.jpg
Elon_Musk | Elon_Musk_003.jpg | Detection Score=0.998496 | Scale=1x
Elon_Musk | Elon_Musk_004.jpg | Detection Score=0.994652 | Scale=1x


  2%|▏         | 2/99 [00:40<33:22, 20.64s/it]

...

Alvaro_Morte | Alvaro_Morte_005.jpg | Detection Score=0.993821 | Scale=1x
Chris_Hemsworth | Chris_Hemsworth_002.jpg | Detection Score=0.985891 | Scale=1x
Chris_Hemsworth | Chris_Hemsworth_003.jpg | Detection Score=0.998169 | Scale=1x
FULL IMAGE USED | Chris_Hemsworth | Chris_Hemsworth_001.jpg
Chris_Hemsworth | Chris_Hemsworth_004.jpg | Detection Score=0.990959 | Scale=1x


 99%|█████████▉| 98/99 [31:24<00:15, 15.59s/it]

Chris_Hemsworth | Chris_Hemsworth_005.jpg | Detection Score=0.934378 | Scale=1x
Miley_Cyrus | Miley_Cyrus_002.jpg | Detection Score=0.996848 | Scale=1x
Miley_Cyrus | Miley_Cyrus_003.jpg | Detection Score=0.999293 | Scale=1x
FULL IMAGE USED | Miley_Cyrus | Miley_Cyrus_001.jpg
Miley_Cyrus | Miley_Cyrus_004.jpg | Detection Score=0.997944 | Scale=1x


100%|██████████| 99/99 [31:38<00:00, 19.18s/it]

Miley_Cyrus | Miley_Cyrus_005.jpg | Detection Score=0.979123 | Scale=1x

Gallery Cropping Done
Successful Samples: 490
Failed Samples: 174


In [57]:
# face crops for probes using RetinaFace with fallback upscaling

import os
import cv2
import pandas as pd

from tqdm import tqdm
from retinaface import RetinaFace

INPUT_ROOT = "natural_test/probes"
OUTPUT_ROOT = "natural_test/probes_cropped"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

failed_images = []
metadata_rows = []

for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)

    if not os.path.isdir(input_dir):
        continue

    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        try:

            img = cv2.imread(image_path)

            if img is None:
                failed_images.append(image_path)
                continue

            h, w = img.shape[:2]

            # ----------------------------------
            # Detection attempts
            # ----------------------------------

            faces = RetinaFace.detect_faces(img)

            scale_used = 1

            if not isinstance(faces, dict) or len(faces) == 0:

                img_2x = cv2.resize(
                    img,
                    None,
                    fx=2,
                    fy=2,
                    interpolation=cv2.INTER_CUBIC,
                )

                faces = RetinaFace.detect_faces(img_2x)

                scale_used = 2

            if not isinstance(faces, dict) or len(faces) == 0:

                img_3x = cv2.resize(
                    img,
                    None,
                    fx=3,
                    fy=3,
                    interpolation=cv2.INTER_CUBIC,
                )

                faces = RetinaFace.detect_faces(img_3x)

                scale_used = 3

            # ----------------------------------
            # No face detected
            # ----------------------------------

            if not isinstance(faces, dict) or len(faces) == 0:

                crop = cv2.resize(img, (112, 112))

                save_path = os.path.join(output_dir, image_file)

                cv2.imwrite(save_path, crop)

                metadata_rows.append(
                    {
                        "identity": identity,
                        "image_name": image_file,
                        "det_score": 0.0,
                        "scale_used": 0,
                    }
                )

                failed_images.append(image_path)

                print(f"FULL IMAGE USED | {identity} | {image_file}")

                continue

            # ----------------------------------
            # Largest face
            # ----------------------------------

            largest_area = -1
            best_box = None
            best_score = None

            for face_id in faces:

                x1, y1, x2, y2 = faces[face_id]["facial_area"]

                area = (x2 - x1) * (y2 - y1)

                if area > largest_area:

                    largest_area = area

                    best_box = (x1, y1, x2, y2)

                    best_score = float(faces[face_id]["score"])

            if best_box is None:

                failed_images.append(image_path)

                continue

            x1, y1, x2, y2 = best_box

            # ----------------------------------
            # Scale coordinates back
            # ----------------------------------

            x1 = int(x1 / scale_used)
            y1 = int(y1 / scale_used)

            x2 = int(x2 / scale_used)
            y2 = int(y2 / scale_used)

            # ----------------------------------
            # Padding
            # ----------------------------------

            pad = 20

            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)

            x2 = min(w, x2 + pad)
            y2 = min(h, y2 + pad)

            crop = img[y1:y2, x1:x2]

            if crop.size == 0:

                crop = cv2.resize(img, (112, 112))

            else:

                crop = cv2.resize(crop, (112, 112))

            save_path = os.path.join(output_dir, image_file)

            cv2.imwrite(save_path, crop)

            metadata_rows.append(
                {
                    "identity": identity,
                    "image_name": image_file,
                    "det_score": best_score,
                    "scale_used": scale_used,
                }
            )

            print(
                f"{identity} | "
                f"{image_file} | "
                f"Detection Score={best_score:.6f} | "
                f"Scale={scale_used}x"
            )

        except Exception as e:

            print(f"\nERROR: {image_path}")
            print(e)

            failed_images.append(image_path)

# ----------------------------------
# Save Logs
# ----------------------------------

pd.DataFrame({"failed_image": failed_images}).to_csv(
    "probe_failed_detections.csv",
    index=False,
)

pd.DataFrame(metadata_rows).to_csv(
    "probe_detection_scores.csv",
    index=False,
)

print("\nProbe Cropping Done")

print("Successful Samples:", len(metadata_rows))
print("Failed Samples:", len(failed_images))

  0%|          | 0/99 [00:00<?, ?it/s]

Zendaya | Screenshot 2026-06-21 154627.png | Detection Score=0.999514 | Scale=1x
Zendaya | Screenshot 2026-06-21 154849.png | Detection Score=0.999587 | Scale=1x


  1%|          | 1/99 [00:09<15:59,  9.79s/it]

Zendaya | Screenshot 2026-06-21 154927.png | Detection Score=0.999473 | Scale=1x
Elon_Musk | Screenshot 2026-06-21 170357.png | Detection Score=0.998835 | Scale=1x
Elon_Musk | Screenshot 2026-06-21 170423.png | Detection Score=0.998911 | Scale=1x


  2%|▏         | 2/99 [00:15<12:00,  7.43s/it]

...

Alvaro_Morte | Screenshot 2026-06-21 181748.png | Detection Score=0.998906 | Scale=1x
Chris_Hemsworth | Screenshot 2026-06-21 151602.png | Detection Score=0.999314 | Scale=1x
Chris_Hemsworth | Screenshot 2026-06-21 151616.png | Detection Score=0.999504 | Scale=1x


 99%|█████████▉| 98/99 [10:30<00:06,  6.66s/it]

Chris_Hemsworth | Screenshot 2026-06-21 151635.png | Detection Score=0.999550 | Scale=1x
Miley_Cyrus | Screenshot 2026-06-21 160939.png | Detection Score=0.999502 | Scale=1x
Miley_Cyrus | Screenshot 2026-06-21 160929.png | Detection Score=0.999613 | Scale=1x


100%|██████████| 99/99 [10:36<00:00,  6.43s/it]

Miley_Cyrus | Screenshot 2026-06-21 161027.png | Detection Score=0.999488 | Scale=1x

Probe Cropping Done
Successful Samples: 294
Failed Samples: 4


In [58]:
for identity in os.listdir("natural_test/gallery"):

    src = os.path.join("natural_test/gallery", identity)
    dst = os.path.join("natural_test/gallery_cropped", identity)

    if not os.path.isdir(src):
        continue

    original = len(os.listdir(src))

    cropped = 0

    if os.path.exists(dst):
        cropped = len(os.listdir(dst))

    print(identity, original, cropped)

Zendaya 5 5
Elon_Musk 5 5
Zac_Efron 5 5
Krysten_Ritter 5 5
Chris_Evans 5 5
Zoe_Saldana 5 5
Barbara_Palvin 5 5
Cristiano_Ronaldo 5 5
Anne_Hathaway 5 5
Madelaine_Petsch 5 5
Josh_Radnor 5 5
Barack_Obama 5 5
Robert_De_Niro 5 5
Nadia_Hilker 5 5
Tom_Holland 5 5
Tom_Ellis 5 5
Natalie_Portman 5 5
Jeff_Bezos 5 5
Alycia_Dabnem_Carey 5 5
Mark_Zuckerberg 5 5
Tom_Hardy 5 5
Ben_Affleck 5 5
Stephen_Amell 5 5
Natalie_Dormer 5 5
Megan_Fox 5 5
Neil_Patrick_Harris 5 5
Avril_Lavigne 5 5
Rebecca_Ferguson 5 5
Shakira 5 5
Melissa_Fumero 5 5
Gal_Gadot 5 5
Selena_Gomez 5 5
Morena_Baccarin 5 5
Tom_Cruise 5 5
Adriana_Lima 5 5
Tom_Hiddleston 5 5
Jessica_Barden 5 5
Leonardo_DiCaprio 5 5
Jimmy_Fallon 5 5
Elizabeth_Lail 5 5
Lili_Reinhart 5 5
Anthony_Mackie 5 5
Gwyneth_Paltrow 5 5
Camila_Mendes 5 5
Margot_Robbie 5 5
Morgan_Freeman 5 5
Grant_Gustin 5 5
Lindsey_Morgan 5 5
Jennifer_Lawrence 5 5
Katherine_Langford 5 5
Emma_Watson 5 5
Lionel_Messi 5 5
Rami_Malek 5 5
Jeremy_Renner 5 5
Mark_Ruffalo 5 5
Sophie_Turner 5 5
Sca

In [59]:
for identity in os.listdir("natural_test/gallery"):

    src = os.path.join("natural_test/gallery", identity)
    dst = os.path.join("natural_test/gallery_cropped", identity)

    if not os.path.isdir(src):
        continue

    cropped_files = set()

    if os.path.exists(dst):
        cropped_files = set(os.listdir(dst))

    for img in os.listdir(src):

        if img not in cropped_files:

            print(identity, img)

In [60]:
# face crops for unknown images
import os
import cv2
from tqdm import tqdm
from insightface.app import FaceAnalysis

INPUT_ROOT = "natural_test/unknown"
OUTPUT_ROOT = "natural_test/unknown_cropped"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)

    # skip .DS_Store and other non-folders
    if not os.path.isdir(input_dir):
        continue

    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        img = cv2.imread(image_path)

        if img is None:
            continue

        faces = app.get(img)

        if len(faces) == 0:
            continue

        best_face = max(faces, key=lambda x: x.det_score)

        bbox = best_face.bbox.astype(int)

        x1, y1, x2, y2 = bbox

        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        save_path = os.path.join(output_dir, image_file)

        cv2.imwrite(save_path, crop)

print("Unknown Cropping Done")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
se

100%|██████████| 3/3 [00:01<00:00,  1.85it/s]

Unknown Cropping Done


In [61]:
# generating arcface embeddings
import os
import cv2
import numpy as np

from tqdm import tqdm
from insightface.app import FaceAnalysis

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

rec_model = app.models["recognition"]


def generate_embeddings(INPUT_ROOT, OUTPUT_ROOT):

    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    count = 0

    for identity in tqdm(os.listdir(INPUT_ROOT)):

        input_dir = os.path.join(INPUT_ROOT, identity)
        output_dir = os.path.join(OUTPUT_ROOT, identity)

        os.makedirs(output_dir, exist_ok=True)

        for image_file in os.listdir(input_dir):

            image_path = os.path.join(input_dir, image_file)

            img = cv2.imread(image_path)

            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (112, 112))

            embedding = rec_model.get_feat(img).flatten().astype(np.float32)

            save_path = os.path.join(
                output_dir, os.path.splitext(image_file)[0] + ".npy"
            )

            np.save(save_path, embedding)

            count += 1

    print("Saved:", count)


generate_embeddings("natural_test/gallery_cropped", "natural_test/gallery_embeddings")

generate_embeddings("natural_test/probes_cropped", "natural_test/probe_embeddings")

generate_embeddings("natural_test/unknown_cropped", "natural_test/unknown_embeddings")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
se

100%|██████████| 98/98 [00:48<00:00,  2.03it/s]


Saved: 490


100%|██████████| 98/98 [00:27<00:00,  3.57it/s]


Saved: 294


100%|██████████| 2/2 [00:00<00:00,  6.81it/s]

Saved: 3


In [62]:
# creating dataset features
import os
import torch
import pyiqa
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

musiq_metric = pyiqa.create_metric("musiq", device=device)


def get_quality_score(image_path):

    score = musiq_metric(image_path)

    return float(score.cpu().item())

Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth


In [63]:
def load_gallery_db(gallery_root):

    gallery_db = {}

    for identity in os.listdir(gallery_root):

        identity_dir = os.path.join(gallery_root, identity)

        # skip .DS_Store
        if not os.path.isdir(identity_dir):
            continue

        embeddings = []

        for file in os.listdir(identity_dir):

            if not file.endswith(".npy"):
                continue

            emb = np.load(os.path.join(identity_dir, file))

            embeddings.append(emb)

        # skip empty identities
        if len(embeddings) == 0:
            continue

        gallery_db[identity] = embeddings

    return gallery_db

In [64]:
# gallery search
def search_gallery(probe_embedding, gallery_db):

    identity_scores = {}

    for identity in gallery_db:

        sims = []

        for gallery_emb in gallery_db[identity]:

            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), gallery_emb.reshape(1, -1)
            )[0][0]

            sims.append(sim)

        identity_scores[identity] = max(sims)

    sorted_scores = sorted(identity_scores.items(), key=lambda x: x[1], reverse=True)

    predicted_identity = sorted_scores[0][0]

    best_similarity = sorted_scores[0][1]

    second_similarity = sorted_scores[1][1]

    margin = best_similarity - second_similarity

    return (predicted_identity, best_similarity, second_similarity, margin)

In [65]:
for identity in os.listdir("natural_test/gallery_embeddings"):

    identity_dir = os.path.join("natural_test/gallery_embeddings", identity)

    if not os.path.isdir(identity_dir):
        continue

    n = len(os.listdir(identity_dir))

    print(identity, n)

Zendaya 5
Elon_Musk 5
Zac_Efron 5
Krysten_Ritter 5
Chris_Evans 5
Zoe_Saldana 5
Barbara_Palvin 5
Cristiano_Ronaldo 5
Anne_Hathaway 5
Madelaine_Petsch 5
Josh_Radnor 5
Barack_Obama 5
Robert_De_Niro 5
Nadia_Hilker 5
Tom_Holland 5
Tom_Ellis 5
Natalie_Portman 5
Jeff_Bezos 5
Alycia_Dabnem_Carey 5
Mark_Zuckerberg 5
Tom_Hardy 5
Ben_Affleck 5
Stephen_Amell 5
Natalie_Dormer 5
Megan_Fox 5
Neil_Patrick_Harris 5
Avril_Lavigne 5
Rebecca_Ferguson 5
Shakira 5
Melissa_Fumero 5
Gal_Gadot 5
Selena_Gomez 5
Morena_Baccarin 5
Tom_Cruise 5
Adriana_Lima 5
Tom_Hiddleston 5
Jessica_Barden 5
Leonardo_DiCaprio 5
Jimmy_Fallon 5
Elizabeth_Lail 5
Lili_Reinhart 5
Anthony_Mackie 5
Gwyneth_Paltrow 5
Camila_Mendes 5
Margot_Robbie 5
Morgan_Freeman 5
Grant_Gustin 5
Lindsey_Morgan 5
Jennifer_Lawrence 5
Katherine_Langford 5
Emma_Watson 5
Lionel_Messi 5
Rami_Malek 5
Jeremy_Renner 5
Mark_Ruffalo 5
Sophie_Turner 5
Scarlett_Johansson 5
Elizabeth_Olsen 5
Dwayne_Johnson 5
Hugh_Jackman 5
Brie_Larson 5
Henry_Cavill 5
Inbar_Lavi 5
Am

In [66]:
# known natural probes dataset
gallery_db = load_gallery_db("natural_test/gallery_embeddings")

rows = []

probe_emb_root = "natural_test/probe_embeddings"
probe_img_root = "natural_test/probes_cropped"

for identity in tqdm(os.listdir(probe_emb_root)):

    emb_dir = os.path.join(probe_emb_root, identity)

    # skip .DS_Store
    if not os.path.isdir(emb_dir):
        continue

    img_dir = os.path.join(probe_img_root, identity)

    for emb_file in os.listdir(emb_dir):

        if not emb_file.endswith(".npy"):
            continue

        probe_embedding = np.load(os.path.join(emb_dir, emb_file))

        base_name = os.path.splitext(emb_file)[0]

        image_path = None

        for ext in [".jpg", ".jpeg", ".png"]:

            candidate = os.path.join(img_dir, base_name + ext)

            if os.path.exists(candidate):

                image_path = candidate
                break

        # image not found
        if image_path is None:
            continue

        # MUSIQ quality score
        quality_score = get_quality_score(image_path)

        # gallery search
        predicted_identity, best_similarity, second_similarity, margin = search_gallery(
            probe_embedding, gallery_db
        )

        # ground truth
        label = int(predicted_identity == identity)

        rows.append(
            {
                "quality_score": quality_score,
                "best_similarity": best_similarity,
                "margin": margin,
                "label": label,
                "true_identity": identity,
                "predicted_identity": predicted_identity,
            }
        )

natural_df = pd.DataFrame(rows)

natural_df.to_csv("natural_probe_dataset.csv", index=False)

print("Saved natural_probe_dataset.csv")
print("Samples:", len(natural_df))

print("\nLabel Distribution")
print(natural_df["label"].value_counts())

100%|██████████| 98/98 [00:53<00:00,  1.84it/s]

Saved natural_probe_dataset.csv
Samples: 294

Label Distribution
label
1    280
0     14
Name: count, dtype: int64


In [67]:
# runing saved svm model
import joblib

svm = joblib.load("svm_rejection_model.pkl")

scaler = joblib.load("svm_scaler.pkl")

df = pd.read_csv("natural_probe_dataset.csv")

X = df[["quality_score", "best_similarity", "margin"]]

X_scaled = scaler.transform(X)

df["svm_decision"] = svm.predict(X_scaled)

df["confidence_score"] = svm.predict_proba(X_scaled)[:, 1]

In [68]:
# svm metrics
from sklearn.metrics import *

y_true = df["label"]
y_pred = df["svm_decision"]

cm = confusion_matrix(y_true, y_pred)

print(cm)

print(classification_report(y_true, y_pred))

TN, FP, FN, TP = cm.ravel()

TAR = TP / (TP + FN)
FRR = FN / (TP + FN)

TRR = TN / (TN + FP)
FAR = FP / (TN + FP)
svm_accuracy = accuracy_score(y_true, y_pred)

svm_TAR = TAR
svm_FRR = FRR
svm_TRR = TRR
svm_FAR = FAR

print("Accuracy:", svm_accuracy)

print("TAR:", svm_TAR)
print("FRR:", svm_FRR)

print("TRR:", svm_TRR)
print("FAR:", svm_FAR)

[[ 14   0]
 [131 149]]
              precision    recall  f1-score   support

           0       0.10      1.00      0.18        14
           1       1.00      0.53      0.69       280

    accuracy                           0.55       294
   macro avg       0.55      0.77      0.44       294
weighted avg       0.96      0.55      0.67       294

Accuracy: 0.5544217687074829
TAR: 0.5321428571428571
FRR: 0.46785714285714286
TRR: 1.0
FAR: 0.0


In [69]:
# fixed threshold method
THRESHOLD = 0.60

threshold_pred = (df["best_similarity"] >= THRESHOLD).astype(int)

cm = confusion_matrix(y_true, threshold_pred)

print(cm)

print(classification_report(y_true, threshold_pred))

TN, FP, FN, TP = cm.ravel()

TAR = TP / (TP + FN)
FRR = FN / (TP + FN)

TRR = TN / (TN + FP)
FAR = FP / (TN + FP)
threshold_accuracy = accuracy_score(y_true, threshold_pred)

threshold_TAR = TAR
threshold_FRR = FRR
threshold_TRR = TRR
threshold_FAR = FAR

print("Accuracy:", threshold_accuracy)

print("TAR:", threshold_TAR)
print("FRR:", threshold_FRR)

print("TRR:", threshold_TRR)
print("FAR:", threshold_FAR)

[[ 14   0]
 [259  21]]
              precision    recall  f1-score   support

           0       0.05      1.00      0.10        14
           1       1.00      0.07      0.14       280

    accuracy                           0.12       294
   macro avg       0.53      0.54      0.12       294
weighted avg       0.95      0.12      0.14       294

Accuracy: 0.11904761904761904
TAR: 0.075
FRR: 0.925
TRR: 1.0
FAR: 0.0


In [70]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# SVM metrics
svm_precision = precision_score(y_true, y_pred)
svm_recall = recall_score(y_true, y_pred)
svm_f1 = f1_score(y_true, y_pred)

# Threshold metrics
threshold_precision = precision_score(y_true, threshold_pred)
threshold_recall = recall_score(y_true, threshold_pred)
threshold_f1 = f1_score(y_true, threshold_pred)

In [71]:
# comparison table
comparison = pd.DataFrame(
    {
        "Method": ["Fixed Threshold", "SVM"],
        "Accuracy": [threshold_accuracy, svm_accuracy],
        "Precision": [threshold_precision, svm_precision],
        "Recall": [threshold_recall, svm_recall],
        "F1": [threshold_f1, svm_f1],
        "TAR": [threshold_TAR, svm_TAR],
        "FRR": [threshold_FRR, svm_FRR],
        "TRR": [threshold_TRR, svm_TRR],
        "FAR": [threshold_FAR, svm_FAR],
    }
)

print(comparison)

            Method  Accuracy  Precision    Recall        F1       TAR  \
0  Fixed Threshold  0.119048        1.0  0.075000  0.139535  0.075000   
1              SVM  0.554422        1.0  0.532143  0.694639  0.532143   

        FRR  TRR  FAR  
0  0.925000  1.0  0.0  
1  0.467857  1.0  0.0  


In [72]:
print("\nNATURAL ACCEPTS")
print(
    natural_df[natural_df["label"] == 1][
        ["best_similarity", "margin", "quality_score"]
    ].describe()
)

print("\nNATURAL REJECTS")
print(
    natural_df[natural_df["label"] == 0][
        ["best_similarity", "margin", "quality_score"]
    ].describe()
)


NATURAL ACCEPTS
       best_similarity      margin  quality_score
count       280.000000  280.000000     280.000000
mean          0.447008    0.218373      38.682915
std           0.093912    0.104906       7.554262
min           0.232499    0.000245      14.602375
25%           0.384853    0.144247      34.869221
50%           0.447680    0.218046      40.087193
75%           0.503557    0.283921      43.987218
max           0.703914    0.542253      54.588047

NATURAL REJECTS
       best_similarity     margin  quality_score
count        14.000000  14.000000      14.000000
mean          0.306303   0.028794      36.456113
std           0.057753   0.021641       9.912012
min           0.200413   0.002699      22.387220
25%           0.275044   0.013254      28.691290
50%           0.315247   0.020152      35.547789
75%           0.336634   0.043926      43.259610
max           0.427181   0.071155      52.521996
